# 05｜正則化與機率分類

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

## 投影片與程式的閱讀方式

對應 `slides/05_正則化與機率分類.md`，目前 43 頁。逐圖完整表見 `programs/SLIDE_NOTEBOOK_MAP_05_25.md`。

L1/L2 幾何、Ridge/Lasso 曲線、早停快照、Sigmoid/交叉熵微分。

各格會標示實驗資料、評估方式與適用範圍；架構圖用於說明機制與張量形狀。

| 程式代碼 | 投影片頁次 | 本格學習內容 |
|---|---|---|
| 05-03 | 3、4、5、7、9、13、15 | 1. Ridge、Lasso、Elastic Net 的係數路徑 |
| 05-04 | 14 | 1. Ridge、Lasso、Elastic Net 的係數路徑 |
| 05-05 | 5、11、15 | 2. 手算驗算：哪組參數的懲罰代價大？ |
| 05-06 | 6、8、10、11、12、43 | 正則化曲線與 L1／L2 幾何 |
| 05-07 | 17、18、43 | 早停：保存並還原驗證最佳參數 |
| 05-08 | 19、23、26、27、30、31 | 3. 手寫 logistic 迴歸：分數 → 機率 → 決策 |
| 05-09 | 20、21、22、23、24、25、26、27、28、29、43 | Sigmoid、交叉熵與第一次更新的驗算 |
| 05-10 | 33、40 | 4. 手算驗算：機率不是最後決策 |
| 05-11 | 32 | 5. 兩個特徵的線性決策邊界 |
| 05-12 | 35、36、37、38 | 6. 手寫穩定 softmax 與多類別交叉熵 |
| 05-13 | 39、41、42 | 7. 用 Pipeline 選正則化強度 C |

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
# 程式 05-01｜投影片 05：2 頁
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4', 'scipy': 'scipy>=1.11,<2', 'sklearn': 'scikit-learn>=1.4,<2'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])

In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
# 程式 05-02｜投影片 05：2 頁
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
if ROOT.name == "notebooks" and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    svg_path = folder / f"{name}.svg"
    plt.gcf().savefig(svg_path, bbox_inches="tight", metadata={"Date": None})
    svg_path.write_text("\n".join(line.rstrip() for line in svg_path.read_text().splitlines()) + "\n")
    plt.show()
    plt.close(plt.gcf())

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

## 1. Ridge、Lasso、Elastic Net 的係數路徑

同一份 12 次多項式特徵，改變正則化。L1 可把部分係數壓成 0；L2 連續縮小。不同 API 的 alpha 尺度不同，數值不可直接互比。

In [ ]:
# 程式 05-03｜投影片 05：3、4、5、7、9、13、15 頁
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import root_mean_squared_error

rng = np.random.default_rng(SEED)
x = rng.uniform(-3, 3, size=(160, 1))
y = 0.5 * x[:, 0] ** 2 - x[:, 0] + 2 + rng.normal(0, 2, 160)
folds = list(KFold(5, shuffle=True, random_state=SEED).split(x))

def poly(est):
    return make_pipeline(PolynomialFeatures(12, include_bias=False), StandardScaler(), est)

models = {'OLS': LinearRegression(), 'Ridge(a=1)': Ridge(alpha=1.0),
          'Lasso(a=.05)': Lasso(alpha=.05, max_iter=100000),
          'ElasticNet(a=.05,r=.5)': ElasticNet(alpha=.05, l1_ratio=.5, max_iter=100000)}
rows = []
for name, est in models.items():
    pipe = poly(est).fit(x, y)
    coef = pipe[-1].coef_
    cv = -cross_val_score(poly(est), x, y, cv=folds, scoring='neg_root_mean_squared_error').mean()
    rows.append({'model': name, 'CV_RMSE': cv, 'L2_norm': np.linalg.norm(coef),
                 'zero_coefs': int(np.sum(np.abs(coef) < 1e-8))})
    plt.plot(np.arange(1, 13), coef, 'o-', label=name)
plt.yscale('symlog', linthresh=.1); plt.xlabel('polynomial power'); plt.ylabel('scaled coefficient'); plt.legend()
savefig('10_regularization_paths')
reg_table = pd.DataFrame(rows).set_index('model')
assert reg_table.loc['Lasso(a=.05)', 'zero_coefs'] > 0 and reg_table.loc['Ridge(a=1)', 'zero_coefs'] == 0
reg_table

In [ ]:
# 程式 05-04｜投影片 05：14 頁
# 同叫 alpha，目標函數尺度不同：教材式 4-13 的 L2 項帶 1/m，sklearn ElasticNet 不帶。
m = len(x)
Zp = make_pipeline(PolynomialFeatures(12, include_bias=False), StandardScaler()).fit_transform(x)
w = ElasticNet(alpha=.05, l1_ratio=.5, max_iter=100000).fit(Zp, y).coef_
mse = np.mean((Zp @ w + y.mean() - y) ** 2)   # 近似資料項
book_obj = mse + 2 * .5 * .05 * np.abs(w).sum() + (1 - .5) * .05 / m * np.sum(w ** 2)
sklearn_obj = mse / 2 + .05 * .5 * np.abs(w).sum() + .05 * (1 - .5) / 2 * np.sum(w ** 2)
print('教材式 4-13 目標 ≈', round(book_obj, 4), ' | sklearn ElasticNet 目標 ≈', round(sklearn_obj, 4))
print('兩者不相等；跨 API 不可使用同一個 alpha。')

## 2. 手算驗算：哪組參數的懲罰代價大？

延伸投影片的 L1／L2 比較（課堂活動單保留手算題）。$w_A=(2,0)$、$w_B=(1,1)$，資料誤差相同。

In [ ]:
# 程式 05-05｜投影片 05：5、11、15 頁
wA, wB = np.array([2., 0.]), np.array([1., 1.])
pen = pd.DataFrame({'L1_norm': [np.abs(wA).sum(), np.abs(wB).sum()],
                    'L2_norm_squared': [np.sum(wA ** 2), np.sum(wB ** 2)]},
                   index=['w_A = (2, 0)', 'w_B = (1, 1)'])
assert pen.L1_norm.tolist() == [2., 2.]                 # L1 相同
assert pen.L2_norm_squared.tolist() == [4., 2.]         # Ridge 偏好 w_B
pen

## 正則化曲線與 L1／L2 幾何

固定合成資料與轉換，只改 alpha，比較 L1 與 L2 的約束幾何。左下約束邊界是等範數集合，不能把外觀直接當成所有資料的最優解。

In [ ]:
# 程式 05-06｜投影片 05：6、8、10、11、12、43 頁
grid_reg=np.linspace(-3,3,300)[:,None]
fig,axes=plt.subplots(2,2,figsize=(11,7))
for ax,estimator in zip(axes[0],[Ridge,Lasso]):
    ax.scatter(x[:,0],y,s=8,alpha=.25)
    for alpha in [.001,.1,10.]:
        est=estimator(alpha=alpha,**({'max_iter':100000} if estimator is Lasso else {}))
        fitted=poly(est).fit(x,y);ax.plot(grid_reg[:,0],fitted.predict(grid_reg),label=f'alpha={alpha}')
    ax.set(title=estimator.__name__,xlabel='x',ylabel='prediction');ax.legend()
a=np.linspace(-2,2,300);aa,bb=np.meshgrid(a,a)
loss_geom=(aa-1.4)**2+2*(bb-.5)**2
for ax,norm,label in [(axes[1,0],abs(aa)+abs(bb),'L1'),(axes[1,1],aa**2+bb**2,'L2 squared')]:
    ax.contour(aa,bb,loss_geom,levels=[.2,.5,1,2,4]);ax.contour(aa,bb,norm,levels=[1],colors='orange',linewidths=3)
    ax.set(title=label,xlabel='w1',ylabel='w2',aspect='equal')
plt.tight_layout();savefig('c05_regularization_geometry')
# |w| 在0的次梯度可取[-1,1]；軟閾值呈現稀疏化的可計算例子。
soft_threshold=lambda value,lam:np.sign(value)*np.maximum(abs(value)-lam,0)
assert np.allclose(soft_threshold(np.array([-.2,0.,.8]),.3),[0,0,.5])

## 早停：保存並還原驗證最佳參數

只用訓練集擬合縮放與多項式。逐 epoch 記錄 training／validation MSE，patience 到期停止；還原的是深複製的最佳模型，不是最後一輪。

In [ ]:
# 程式 05-07｜投影片 05：17、18、43 頁
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import train_test_split
from sklearn.base import clone
import copy
es_xtr,es_xval,es_ytr,es_yval=train_test_split(x,y,test_size=.3,random_state=SEED)
es_prep=make_pipeline(PolynomialFeatures(12,include_bias=False),StandardScaler()).fit(es_xtr)
es_train=es_prep.transform(es_xtr);es_val=es_prep.transform(es_xval)
es=SGDRegressor(penalty=None,learning_rate='constant',eta0=.001,random_state=SEED)
history=[];best=np.inf;wait=0;best_model=None;patience=25
for epoch in range(500):
    es.partial_fit(es_train,es_ytr)
    train_mse=np.mean((es.predict(es_train)-es_ytr)**2);val_mse=np.mean((es.predict(es_val)-es_yval)**2)
    history.append([train_mse,val_mse])
    if val_mse<best:
        best=val_mse;best_epoch=epoch;best_model=copy.deepcopy(es);wait=0
    else:wait+=1
    if wait>=patience:break
assert np.isclose(np.mean((best_model.predict(es_val)-es_yval)**2),best)
plt.plot(history);plt.axvline(best_epoch,color='k',ls='--',label='best validation epoch')
plt.xlabel('Epoch');plt.ylabel('MSE');plt.legend(['train','validation','best epoch']);savefig('c05_early_stopping')
print('Stopped / best epoch:',len(history),best_epoch+1,'best validation MSE:',best)

## 3. 手寫 logistic 迴歸：分數 → 機率 → 決策

以 Iris virginica 為正類、花瓣寬度為單一特徵。用 `expit` 與 `logaddexp` 保持數值穩定；梯度 $\nabla J = \frac1m X^\top(\hat p - y)$，沒有 MSE 的係數 2。

In [ ]:
# 程式 05-08｜投影片 05：19、23、26、27、30、31 頁
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from scipy.special import expit
iris = load_iris()
Xpw = iris.data[:, 3:4]                       # 花瓣寬度
y_vir = (iris.target == 2).astype(float)      # virginica = 正類
Xtr, Xte, ytr, yte = train_test_split(Xpw, y_vir, test_size=.3, stratify=y_vir, random_state=SEED)
mu, sd = Xtr.mean(), Xtr.std()
Ztr, Zte = (Xtr - mu) / sd, (Xte - mu) / sd

def fit_logistic(Z, y, eta=.3, epochs=800):
    w, b, loss = 0.0, 0.0, []
    for _ in range(epochs):
        z = Z[:, 0] * w + b
        p = expit(z)
        loss.append(np.mean(np.logaddexp(0, z) - y * z))       # 穩定的平均交叉熵
        g = p - y
        w -= eta * np.mean(g * Z[:, 0]); b -= eta * np.mean(g)
    return w, b, np.array(loss)

w, b, loss = fit_logistic(Ztr, ytr)
grid = np.linspace(Xpw.min(), Xpw.max(), 200)
prob = expit(((grid - mu) / sd) * w + b)
boundary = grid[np.argmin(np.abs(prob - 0.5))]
plt.plot(grid, prob, label='P(virginica)')
plt.axhline(.5, ls=':', c='grey'); plt.axvline(boundary, ls='--', c='k', label=f'0.5 boundary ≈ {boundary:.2f} cm')
plt.scatter(Xpw[:, 0], y_vir, c=y_vir, cmap='RdBu_r', edgecolor='k', alpha=.6)
plt.xlabel('petal width (cm)'); plt.ylabel('probability / label'); plt.legend()
savefig('10_logistic_probability')
assert np.all(np.diff(loss) <= 1e-9)          # 交叉熵單調下降
assert 1.4 < boundary < 1.9
print('final training loss:', round(loss[-1], 4), ' decision boundary:', round(boundary, 3), 'cm')

## Sigmoid、交叉熵與第一次更新的驗算

逐步驗算 $\sigma'(z)=p(1-p)$、$\partial\ell/\partial p=(p-y)/(p(1-p))$、$\partial\ell/\partial z=p-y$。用中心差分檢查，並重算投影片的兩筆手算例。

In [ ]:
# 程式 05-09｜投影片 05：20、21、22、23、24、25、26、27、28、29、43 頁
zg=np.linspace(-6,6,200);pg=expit(zg)
fig,axes=plt.subplots(1,2,figsize=(10,3.5));axes[0].plot(zg,pg);axes[1].plot(zg,pg*(1-pg))
axes[0].set(xlabel='z',ylabel='sigmoid');axes[1].set(xlabel='z',ylabel='sigmoid derivative');savefig('c05_sigmoid_derivative')
z0=np.array([-.7,.2,1.3]);y0=np.array([0.,1.,1.]);p0=expit(z0)
dp=(p0-y0)/(p0*(1-p0));assert np.allclose(dp*p0*(1-p0),p0-y0)
loss_z=lambda z:np.logaddexp(0,z)-y0*z
h=1e-5;assert np.allclose((loss_z(z0+h)-loss_z(z0-h))/(2*h),p0-y0)
Xd=np.array([[1.,0.],[1.,1.]]);yd=np.array([0.,1.]);th=np.zeros(2)
prob0=expit(Xd@th);g=Xd.T@(prob0-yd)/len(yd);new=th-g
objective=lambda t:np.mean(np.logaddexp(0,Xd@t)-yd*(Xd@t))
assert np.allclose(g,[0,-.25]) and np.allclose(new,[0,.25])
assert np.isclose(objective(new),.6345433002193944)
assert np.mean((prob0>=.5)==yd)==np.mean((expit(Xd@new)>=.5)==yd)==.5
likelihood=np.prod(prob0**yd*(1-prob0)**(1-yd));assert np.isclose(-np.log(likelihood)/2,objective(th))
pd.Series({'gradient':g.tolist(),'new_theta':new.tolist(),'loss_before':objective(th),'loss_after':objective(new),'accuracy_before_after':.5})

## 4. 手算驗算：機率不是最後決策

對應投影片活動「比較兩個閾值」。真值 `[1,0,1,0]`、正類機率 `[.8,.6,.4,.1]`。

In [ ]:
# 程式 05-10｜投影片 05：33、40 頁
probs = np.array([.8, .6, .4, .1]); truth = np.array([1, 0, 1, 0])
def counts(t):
    pred = probs >= t
    tp = int(np.sum(pred & (truth == 1))); fp = int(np.sum(pred & (truth == 0)))
    fn = int(np.sum(~pred & (truth == 1))); tn = int(np.sum(~pred & (truth == 0)))
    return {'threshold': t, 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn}
thr = pd.DataFrame([counts(.5), counts(.3)])
assert thr[['TP', 'FP', 'FN', 'TN']].values.tolist() == [[1, 1, 1, 1], [2, 1, 0, 1]]
thr

## 5. 兩個特徵的線性決策邊界

logistic 對原始特徵的分數是線性的，決策邊界是直線。加入非線性特徵才會彎曲。

此段用全部 Iris 畫訓練資料的幾何示意，不報告泛化成效；最後一節才使用分割、CV 與獨立測試。

In [ ]:
# 程式 05-11｜投影片 05：32 頁
from sklearn.linear_model import LogisticRegression
X2 = iris.data[:, 2:4]                        # 花瓣長、寬
lr2 = make_pipeline(StandardScaler(), LogisticRegression()).fit(X2, y_vir)
xx, yy = np.meshgrid(np.linspace(X2[:, 0].min() - .5, X2[:, 0].max() + .5, 200),
                     np.linspace(X2[:, 1].min() - .5, X2[:, 1].max() + .5, 200))
zz = lr2.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)
plt.contourf(xx, yy, zz, levels=np.linspace(0, 1, 11), cmap='RdBu_r', alpha=.7)
plt.colorbar(label='P(virginica), uncalibrated')
plt.contour(xx, yy, zz, levels=[.5], colors='k')
plt.scatter(X2[:, 0], X2[:, 1], c=y_vir, cmap='RdBu_r', edgecolor='k', s=25)
plt.xlabel('petal length (cm)'); plt.ylabel('petal width (cm)')
savefig('10_logistic_boundary_2d')

## 6. 手寫穩定 softmax 與多類別交叉熵

softmax 對所有分數加同一常數不變，因此先減最大分數避免溢位。搭配交叉熵時梯度簡化為 $\hat p - y$。

三類決策區域使用全部 Iris 作訓練圖，印出的 accuracy 明確是訓練分數；正式評估在下一節另行切分。

In [ ]:
# 程式 05-12｜投影片 05：35、36、37、38 頁
def softmax(s):
    s = np.asarray(s, float)
    e = np.exp(s - s.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

assert np.allclose(softmax([1000, 1001, 1002]), softmax([-2, -1, 0]))     # 平移不變
demo = softmax([0, 1, 2])
assert np.allclose(demo, [0.09003057, 0.24472847, 0.66524096], atol=1e-6)
assert np.isclose(-np.log(demo[2]), 0.40760596, atol=1e-6)                # 類別 2 的損失
print('softmax([0,1,2]) =', np.round(demo, 5), ' | class-2 loss =', round(float(-np.log(demo[2])), 5))

# 三類 Iris：softmax 迴歸的決策區域（教材圖 4-26），用兩個花瓣特徵
Xs = StandardScaler().fit_transform(iris.data[:, 2:4]); ys = iris.target
W = np.zeros((3, 3))                          # 3 類 × (bias + 2 特徵)
Xa = np.c_[np.ones(len(Xs)), Xs]
Y = np.eye(3)[ys]
for _ in range(4000):
    P = softmax(Xa @ W.T)
    W -= 0.1 * ((P - Y).T @ Xa) / len(Xa)
pred = np.argmax(Xa @ W.T, axis=1)
zz = np.argmax(np.c_[np.ones(xx.size), ((np.c_[xx.ravel(), yy.ravel()] - iris.data[:, 2:4].mean(0)) /
     iris.data[:, 2:4].std(0))] @ W.T, axis=1).reshape(xx.shape)
plt.contourf(xx, yy, zz, levels=[-.5, .5, 1.5, 2.5], cmap='Pastel1', alpha=.8)
plt.scatter(iris.data[:, 2], iris.data[:, 3], c=ys, cmap='Dark2', edgecolor='k', s=25)
plt.xlabel('petal length (cm)'); plt.ylabel('petal width (cm)')
savefig('10_softmax_regions')
print('hand-written softmax regression training accuracy:', round(float(np.mean(pred == ys)), 3))

## 7. 用 Pipeline 選正則化強度 C

`C` 越小正則化越強。標準化放進 Pipeline，`GridSearchCV` 每折重新 fit；選定後才看一次測試。

In [ ]:
# 程式 05-13｜投影片 05：39、41、42 頁
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import confusion_matrix, log_loss
Xtr, Xte, ytr, yte = train_test_split(iris.data, iris.target, test_size=.3, stratify=iris.target, random_state=SEED)
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
search = GridSearchCV(pipe, {'logisticregression__C': [0.01, 0.1, 1, 10]}, cv=5).fit(Xtr, ytr)
final = search.best_estimator_
proba = final.predict_proba(Xte)
metrics = {'best_C': search.best_params_['logisticregression__C'],
           'CV_accuracy': float(search.best_score_),
           'test_accuracy': float(final.score(Xte, yte)),
           'test_log_loss': float(log_loss(yte, proba)),
           'test_confusion_matrix': confusion_matrix(yte, final.predict(Xte)).tolist(),
           'softmax_demo': demo.tolist(), 'softmax_class2_loss': float(-np.log(demo[2]))}
report('regularization_classification', metrics)
pd.Series(metrics)

## 練習與參考答案

- 係數變小代表特徵沒有因果作用嗎？**否，只表示在此正則化與尺度下權重被壓縮。**
- 驗證集選過 `C` 或早停後還能當測試集嗎？**不能，它已參與模型選擇。**
- softmax 機率加總為 1 代表已校準嗎？**否，加總為 1 是定義，校準要另外檢查。**